clean Data 

In [1]:
"""
Stock Trend Analysis Script
Analyzes closing price and volume trends for multiple stocks.
Supports: TCS, RELIANCE, MSFT, HLL, HDBK (or any similarly formatted CSV)

CSV format expected:
  "Date","Price","Open","High","Low","Vol.","Change %"
  "29-04-2026","778.40","785.50","789.25","777.50","25.80M","-0.53%"
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os
from pathlib import Path

# ─────────────────────────────────────────────
# 1. CONFIGURATION — update file paths here
# ─────────────────────────────────────────────


CSV_FILES = {
    "HLL":      r"E:\KEC TASK\Download data\HLL Historical Data.csv",
    "TCS":      r"E:\KEC TASK\Download data\TCS Historical Data.csv",
    "RELIANCE": r"E:\KEC TASK\Download data\RELI Historical Data.csv",
    "MSFT":     r"E:\KEC TASK\Download data\MSFT Historical Data.csv",
    "HDBK":     r"E:\KEC TASK\Download data\HDBK Historical Data (1).csv",
}

OUTPUT_DIR = "output_charts_data_analysis"   # folder where charts are saved
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────
# 2. HELPER FUNCTIONS
# ─────────────────────────────────────────────

def parse_volume(vol_str):
    """Convert volume strings like '25.80M', '1.5K', '2B' to float."""
    vol_str = str(vol_str).strip().upper().replace(",", "")
    if vol_str in ("", "N/A", "NAN", "-"):
        return np.nan
    multipliers = {"K": 1e3, "M": 1e6, "B": 1e9}
    for suffix, mult in multipliers.items():
        if vol_str.endswith(suffix):
            return float(vol_str[:-1]) * mult
    return float(vol_str)


def load_stock(filepath, ticker):
    """Load and clean a single stock CSV file."""
    if not Path(filepath).exists():
        print(f"  [SKIP] File not found: {filepath}")
        return None

    df = pd.read_csv(filepath)

    # Standardise column names (strip whitespace/quotes)
    df.columns = [c.strip().strip('"') for c in df.columns]

    # Parse date
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.sort_values("Date")

    # Parse numeric columns (remove commas)
    for col in ["Price", "Open", "High", "Low"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(",", "").astype(float)

    # Parse volume
    if "Vol." in df.columns:
        df["Volume"] = df["Vol."].apply(parse_volume)

    # Parse change %
    if "Change %" in df.columns:
        df["Change_pct"] = (
            df["Change %"].astype(str)
            .str.replace("%", "")
            .str.replace(",", "")
            .astype(float)
        )

    df["Ticker"] = ticker
    print(f"  [OK] {ticker}: {len(df)} rows  |  {df['Date'].min().date()} → {df['Date'].max().date()}")
    return df


# ─────────────────────────────────────────────
# 3. LOAD ALL FILES
# ─────────────────────────────────────────────
print("\n📂 Loading CSV files...")
stocks = {}
for ticker, filepath in CSV_FILES.items():
    df = load_stock(filepath, ticker)
    if df is not None:
        stocks[ticker] = df

if not stocks:
    raise SystemExit("No CSV files loaded. Check file paths in CSV_FILES.")

all_data = pd.concat(stocks.values(), ignore_index=True)


# ─────────────────────────────────────────────
# 4. SUMMARY STATISTICS
# ─────────────────────────────────────────────
print("\n📊 Summary Statistics (Closing Price)\n" + "=" * 55)
summary_rows = []
for ticker, df in stocks.items():
    p = df["Price"]
    first, last = p.iloc[0], p.iloc[-1]
    pct_return = ((last - first) / first) * 100
    summary_rows.append({
        "Ticker":      ticker,
        "Rows":        len(df),
        "Start Price": round(first, 2),
        "End Price":   round(last, 2),
        "Min":         round(p.min(), 2),
        "Max":         round(p.max(), 2),
        "Mean":        round(p.mean(), 2),
        "Std Dev":     round(p.std(), 2),
        "Return %":    round(pct_return, 2),
    })

summary = pd.DataFrame(summary_rows).set_index("Ticker")
print(summary.to_string())
summary.to_csv(os.path.join(OUTPUT_DIR, "summary_statistics.csv"))
print(f"\n  ✅ Saved: {OUTPUT_DIR}/summary_statistics.csv")


# ─────────────────────────────────────────────
# 5. MOVING AVERAGES (5-day & 10-day)
# ─────────────────────────────────────────────
for ticker, df in stocks.items():
    df["MA5"]  = df["Price"].rolling(window=5,  min_periods=1).mean()
    df["MA10"] = df["Price"].rolling(window=10, min_periods=1).mean()


# ─────────────────────────────────────────────
# 6. CHART 1 — Closing Price Trend (all stocks)
# ─────────────────────────────────────────────
print("\n📈 Generating charts...")

fig, ax = plt.subplots(figsize=(14, 6))
colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]

for (ticker, df), color in zip(stocks.items(), colors):
    ax.plot(df["Date"], df["Price"], marker="o", markersize=4,
            linewidth=2, label=ticker, color=color)
    ax.plot(df["Date"], df["MA5"],  linestyle="--", linewidth=1,
            alpha=0.6, color=color)

ax.set_title("Closing Price Trend — All Stocks", fontsize=15, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator())
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
path1 = os.path.join(OUTPUT_DIR, "01_closing_price_trend.png")
plt.savefig(path1, dpi=150)
plt.close()
print(f"  ✅ Saved: {path1}")


# ─────────────────────────────────────────────
# 7. CHART 2 — Volume Trend (all stocks)
# ─────────────────────────────────────────────
fig, axes = plt.subplots(len(stocks), 1,
                         figsize=(14, 3.5 * len(stocks)), sharex=False)
if len(stocks) == 1:
    axes = [axes]

for ax, ((ticker, df), color) in zip(axes, zip(stocks.items(), colors)):
    if "Volume" in df.columns:
        ax.bar(df["Date"], df["Volume"] / 1e6, color=color, alpha=0.75, width=0.8)
        ax.set_title(f"{ticker} — Daily Volume (Millions)", fontweight="bold")
        ax.set_ylabel("Volume (M)")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
        plt.setp(ax.get_xticklabels(), rotation=45)
        ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("Volume Trend — All Stocks", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
path2 = os.path.join(OUTPUT_DIR, "02_volume_trend.png")
plt.savefig(path2, dpi=150, bbox_inches="tight")
plt.close()
print(f"  ✅ Saved: {path2}")


# ─────────────────────────────────────────────
# 8. CHART 3 — Daily % Change Heatmap-style bar
# ─────────────────────────────────────────────
if all("Change_pct" in df.columns for df in stocks.values()):
    fig, axes = plt.subplots(len(stocks), 1,
                             figsize=(14, 2.8 * len(stocks)), sharex=False)
    if len(stocks) == 1:
        axes = [axes]

    for ax, (ticker, df) in zip(axes, stocks.items()):
        bar_colors = ["#4CAF50" if v >= 0 else "#F44336" for v in df["Change_pct"]]
        ax.bar(df["Date"], df["Change_pct"], color=bar_colors, width=0.7)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(f"{ticker} — Daily Change %", fontweight="bold")
        ax.set_ylabel("Change %")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
        plt.setp(ax.get_xticklabels(), rotation=45)
        ax.grid(True, axis="y", alpha=0.3)

    plt.suptitle("Daily % Change — All Stocks", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    path3 = os.path.join(OUTPUT_DIR, "03_daily_change_pct.png")
    plt.savefig(path3, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✅ Saved: {path3}")


# ─────────────────────────────────────────────
# 9. CHART 4 — Normalised price (base = 100)
#    Great for comparing performance across stocks
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for (ticker, df), color in zip(stocks.items(), colors):
    base = df["Price"].iloc[0]
    normalised = (df["Price"] / base) * 100
    ax.plot(df["Date"], normalised, marker="o", markersize=4,
            linewidth=2, label=ticker, color=color)

ax.axhline(100, color="grey", linestyle="--", linewidth=1, alpha=0.6)
ax.set_title("Normalised Price Performance (Base = 100)", fontsize=15, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Normalised Price")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator())
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
path4 = os.path.join(OUTPUT_DIR, "04_normalised_performance.png")
plt.savefig(path4, dpi=150)
plt.close()
print(f"  ✅ Saved: {path4}")


# ─────────────────────────────────────────────
# 10. INDIVIDUAL STOCK CHARTS (Price + Volume)
# ─────────────────────────────────────────────
for (ticker, df), color in zip(stocks.items(), colors):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
                                    gridspec_kw={"height_ratios": [3, 1]},
                                    sharex=True)

    # Price + MAs
    ax1.plot(df["Date"], df["Price"], color=color,   linewidth=2,   label="Close")
    ax1.plot(df["Date"], df["MA5"],   color="orange", linewidth=1.2, linestyle="--", label="MA5")
    ax1.plot(df["Date"], df["MA10"],  color="purple", linewidth=1.2, linestyle=":",  label="MA10")
    ax1.fill_between(df["Date"], df["Low"], df["High"], alpha=0.1, color=color)
    ax1.set_title(f"{ticker} — Price & Volume", fontsize=13, fontweight="bold")
    ax1.set_ylabel("Price")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Volume
    if "Volume" in df.columns:
        ax2.bar(df["Date"], df["Volume"] / 1e6, color=color, alpha=0.7, width=0.7)
        ax2.set_ylabel("Vol (M)")
        ax2.grid(True, axis="y", alpha=0.3)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    plt.xticks(rotation=45)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"05_{ticker}_detail.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  ✅ Saved: {path}")


# ─────────────────────────────────────────────
# 11. FULL DETAIL CSV (all stocks combined)
# ─────────────────────────────────────────────
all_data.to_csv(os.path.join(OUTPUT_DIR, "all_stocks_combined.csv"), index=False)
print(f"\n  ✅ Saved: {OUTPUT_DIR}/all_stocks_combined.csv")

print("\n✅ All done! Check the '{}' folder for charts and CSVs.\n".format(OUTPUT_DIR))


📂 Loading CSV files...
  [OK] HLL: 20 rows  |  2026-03-30 → 2026-04-29
  [OK] TCS: 20 rows  |  2026-03-30 → 2026-04-29
  [OK] RELIANCE: 20 rows  |  2026-03-30 → 2026-04-29
  [OK] MSFT: 21 rows  |  2026-03-30 → 2026-04-28
  [OK] HDBK: 834 rows  |  2021-07-12 → 2026-04-29

📊 Summary Statistics (Closing Price)
          Rows  Start Price  End Price      Min      Max     Mean  Std Dev  Return %
Ticker                                                                              
HLL         20      2055.20    2317.10  2055.20  2368.80  2200.95   108.67     12.74
TCS         20      2358.90    2465.00  2358.90  2610.50  2504.72    72.21      4.50
RELIANCE    20      1343.90    1430.00  1304.60  1430.00  1350.15    28.54      6.41
MSFT        21       358.96     429.25   358.96   432.92   396.99    25.91     19.58
HDBK       834       741.65     778.40   639.06  1012.90   830.86   103.14      4.96

  ✅ Saved: output_charts_data_analysis/summary_statistics.csv

📈 Generating charts...
  ✅ Save

Price Trend (All Stocks)
 Shows how prices change over time
 Includes MA5


Volume Trend
 Shows trading activity
 Uses bar charts


Daily % Change
 Green → positive
 Red → negative


Individual Stock Analysis
 Each stock has:
 Price trend
 Moving averages
 High–Low range
 Volume


OUTPUT
 Price trend
 Volume trend
 Daily change
 Normalized performance
 Individual stock charts


 output will generated in output_charts_data_analysis this  folder
